# 04 — Monitorización, logs centralizados y alertas automáticas

Este notebook explica Loki, Promtail, `system_events`, `alerts` y el servicio `automation`.

In [ ]:

from pathlib import Path
import json
import os
import subprocess
import textwrap
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path | None = None) -> Path:
    """Busca la raíz del proyecto localizando docker-compose.yml."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "docker-compose.yml").exists():
            return candidate
    # Si el notebook se ejecuta desde notebooks/ antes de abrir el repo correctamente.
    return start


ROOT = find_project_root()
print(f"ROOT = {ROOT}")


def read_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_env(path: Path) -> dict:
    env = {}
    path = Path(path)
    if not path.exists():
        return env
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env


def run_cmd(cmd: str, timeout: int = 60) -> str:
    """Ejecuta un comando shell y devuelve stdout/stderr como texto."""
    print(f"$ {cmd}")
    try:
        result = subprocess.run(
            cmd,
            shell=True,
            cwd=ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        print(f"exit_code={result.returncode}")
        return (result.stdout or "") + (result.stderr or "")
    except Exception as exc:
        print(f"[error] {exc}")
        return str(exc)


def show_bar(series, title: str, ylabel: str = "valor"):
    ax = pd.Series(series).plot(kind="bar")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

print("=== Monitorización y calidad de datos ===")

monitoring_files = [
    ROOT / "monitoring" / "loki-config.yaml",
    ROOT / "monitoring" / "promtail-config.yaml",
    ROOT / "services" / "pipeline" / "app" / "logging_setup.py",
    ROOT / "services" / "pipeline" / "app" / "events.py",
    ROOT / "services" / "automation" / "main.py",
]

rows = []
for p in monitoring_files:
    rows.append({
        "archivo": str(p.relative_to(ROOT)),
        "existe": p.exists(),
        "tamaño_kb": round(p.stat().st_size / 1024, 2) if p.exists() else None,
    })
display(pd.DataFrame(rows))

print("\n=== Estado Loki / Promtail / Automation ===")
run_cmd("docker compose ps loki promtail automation", timeout=60)

print("\n=== Loki ready ===")
run_cmd("curl -s http://localhost:3100/ready", timeout=30)

print("\n=== Últimos logs de automation ===")
run_cmd("docker compose logs automation --tail=30", timeout=60)

print("\n=== Consulta ejemplo a MongoDB: eventos warning/error ===")
run_cmd(
    'docker compose exec -T mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --quiet --eval "db.system_events.find({level:{\\$in:[\'warning\',\'error\']}}).sort({timestamp:-1}).limit(5).forEach(printjson)"',
    timeout=60
)

print("\n=== Consulta ejemplo a MongoDB: alertas ===")
run_cmd(
    'docker compose exec -T mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --quiet --eval "db.alerts.find().sort({created_at:-1}).limit(5).forEach(printjson)"',
    timeout=60
)

## 1. Capas de observabilidad

El proyecto tiene varias capas:

| Capa | Tecnología | Función |
|---|---|---|
| Logs técnicos | stdout JSON | Logs de servicios. |
| Centralización | Loki + Promtail | Recolectar logs Docker. |
| Eventos de dominio | MongoDB `system_events` | Eventos funcionales del pipeline. |
| Alertas | MongoDB `alerts` | Eventos warning/error convertidos en alertas. |
| Dashboard | Streamlit / Mongo Express | Visualización. |

In [ ]:
monitoring_files = [
    "monitoring/loki-config.yaml",
    "monitoring/promtail-config.yaml",
    "services/automation/main.py",
    "services/pipeline/app/events.py",
    "services/pipeline/app/logging_setup.py",
]
pd.DataFrame([{"archivo": p, "existe": (ROOT/p).exists()} for p in monitoring_files])


## 2. Loki y Promtail

Loki almacena logs. Promtail recoge logs de Docker.

Validación usada:

```powershell
curl.exe -s http://localhost:3100/ready
```

Consulta LogQL real:

```powershell
$query = '{compose_service="pipeline"} |= "pipeline.run.end"'
$url = "http://localhost:3100/loki/api/v1/query_range?limit=10&start=$start&end=$end&query=$([uri]::EscapeDataString($query))"
curl.exe -s $url
```

Se considera validado cuando Loki devuelve un evento real `pipeline.run.end`.

In [ ]:
print("Comprobar Loki:")
print("curl.exe -s http://localhost:3100/ready")
print()
print("Labels:")
print("curl.exe -s http://localhost:3100/loki/api/v1/labels")


## 3. Eventos de dominio en MongoDB

La colección `system_events` guarda eventos funcionales del pipeline.

Ejemplo:

```json
{
  "event": "pipeline.validation.done",
  "level": "warning",
  "message": "Validación: 1 válidos, 4 rechazados"
}
```

In [ ]:
event_examples = pd.DataFrame([
    {"event": "pipeline.run.start", "level": "info", "count_demo": 1},
    {"event": "pipeline.file.read", "level": "info", "count_demo": 1},
    {"event": "pipeline.validation.done", "level": "warning", "count_demo": 1},
    {"event": "pipeline.rejects.persisted", "level": "warning", "count_demo": 1},
    {"event": "pipeline.run.end", "level": "info", "count_demo": 1},
])
event_examples


In [ ]:
level_counts = event_examples.groupby("level")["count_demo"].sum()
show_bar(level_counts, "Eventos por nivel — ejemplo de ejecución con rechazos", "eventos")


## 4. Automation

El servicio `automation` ejecuta un loop:

```text
cada N segundos
    ↓
lee system_events recientes
    ↓
filtra warning/error
    ↓
crea dedup_key
    ↓
inserta/actualiza alerts
    ↓
loggea nueva alerta
```

Ejemplo de alerta:

```json
{
  "source_event": "pipeline.validation.done",
  "severity": "warning",
  "status": "open",
  "notification_channel": "mongo_dashboard",
  "notification_status": "simulated"
}
```

In [ ]:
alert_examples = pd.DataFrame([
    {"source_event": "pipeline.validation.done", "severity": "warning", "status": "open"},
    {"source_event": "pipeline.rejects.persisted", "severity": "warning", "status": "open"},
])
alert_examples


## 5. Comandos de consulta

Ver alertas:

```powershell
docker compose exec mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --eval "db.alerts.find().sort({created_at:-1}).limit(10).pretty()"
```

Ver eventos warning/error:

```powershell
docker compose exec mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --eval "db.system_events.find({level:{`$in:['warning','error']}}).sort({timestamp:-1}).limit(10).pretty()"
```

Ver logs automation:

```powershell
docker compose logs automation --tail=50
```

## 6. Qué requisito cubre

Esta parte cubre:

- logging centralizado;
- validación de calidad de datos;
- alertas o notificaciones ante fallos del procesamiento;
- trazabilidad con `correlation_id`;
- auditoría de eventos funcionales.